In [1]:
import os

In [2]:
file_zip = os.path.join(os.getcwd(),os.listdir()[1])

In [3]:
os.chdir("../")

In [4]:
%pwd

'e:\\Samay\\projects\\adenocarcenoma\\adenocarcenoma-classificaation-end-to-end-using-mlflow-and-DVC'

step 4 of workflow
update entity

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
print(type(DataIngestionConfig))

<class 'type'>


step 5 of workflow from readme.md

In [7]:
from cnn_classifier.constants import *
from cnn_classifier.utils.common import read_yaml, create_directories



In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config
      

Data Ingestion Component

Workflow line 6

In [9]:
import os
import zipfile
import gdown
from cnn_classifier import logger
from cnn_classifier.utils.common import get_size

In [10]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
     
    def download_file(self)-> str:
        '''
        Fetch data from the url
        '''

        try: 
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs("artifacts/data_ingestion", exist_ok=True)
            logger.info(f"Downloading data from {dataset_url} into file {zip_download_dir}")

            file_id = dataset_url.split("/")[-2]
            prefix = 'https://drive.google.com/uc?/export=download&id='
            gdown.download(prefix+file_id,zip_download_dir)

            logger.info(f"Downloaded data from {dataset_url} into file {zip_download_dir}")

        except Exception as e:
            raise e
        
    
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)




Pipeline

In [12]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2025-08-31 02:00:49,450: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-08-31 02:00:49,477: INFO: common: yaml file: params.yaml loaded successfully]
[2025-08-31 02:00:49,483: INFO: common: created directory at: artifacts]
[2025-08-31 02:00:49,491: INFO: common: created directory at: artifacts/data_ingestion]
[2025-08-31 02:00:49,498: INFO: 586056171: Downloading data from https://drive.google.com/file/d/1Qqg19dtvL5YM1JcGA_Fan1T0PQuEuf7o/view?usp=drive_link into file artifacts/data_ingestion/data.zip]


Downloading...
From (original): https://drive.google.com/uc?/export=download&id=1Qqg19dtvL5YM1JcGA_Fan1T0PQuEuf7o
From (redirected): https://drive.google.com/uc?%2Fexport=download&id=1Qqg19dtvL5YM1JcGA_Fan1T0PQuEuf7o&confirm=t&uuid=fa657bfa-3fda-4a38-acfc-607fbad50076
To: e:\Samay\projects\adenocarcenoma\adenocarcenoma-classificaation-end-to-end-using-mlflow-and-DVC\artifacts\data_ingestion\data.zip
100%|██████████| 124M/124M [02:01<00:00, 1.02MB/s] 

[2025-08-31 02:02:56,043: INFO: 586056171: Downloaded data from https://drive.google.com/file/d/1Qqg19dtvL5YM1JcGA_Fan1T0PQuEuf7o/view?usp=drive_link into file artifacts/data_ingestion/data.zip]
